# Figure 1d/1e — full-barcode codebook spot detection

This publication notebook is a cleaned duplicate of the original v4 working
spot-detection notebook.

Scope:

- `reg000`: PBS spleen
- `reg001`: SM-102 LNP-treated spleen
- full 12-bit barcode decoding
- cell assignment and CSV summaries used by Figure 1d/1e

The liver region, luciferase/transfection analysis, saved QC images, and unrelated
exploratory outputs have been removed. The large registered TIFF stacks are read
with memory mapping. This notebook is intentionally distributed unexecuted.


## 1. Environment and packaged inputs

Install the small Python dependency set listed in
`Figure_1d_1e_requirements.txt`.

Packaged inputs are under:

`../Data/Figure_1d_1e_Spleen_LNP/Raw_Spot_Detection_Input/`

The two registered multichannel TIFFs are the stitched/registered image products
used for spot detection. `reg000_features.csv` and `reg001_features.csv` contain
the previously generated cell centroids and measurements. Cell segmentation and
cell-type annotation were performed with the previously published workflow and
are treated here as frozen upstream inputs; those upstream notebooks are not
reproduced in this package.

Running the notebook writes CSV files only to
`../Data/Figure_1d_1e_Spleen_LNP/Generated_Output/`.


In [ ]:
from IPython.display import display
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import json
import logging
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy import ndimage as ndi
from scipy.spatial import cKDTree


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger('spot-yining-v4')


def tic():
    return time.perf_counter()


def toc(t0, label=''):
    dt = time.perf_counter() - t0
    log.info(f'{label} done in {dt:.2f} s')
    return dt



def resolve_data_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        for candidate in (
            base / "Manuscripts" / "NanoSTAMP" / "Data" / "Figure_1d_1e_Spleen_LNP",
            base / "Data" / "Figure_1d_1e_Spleen_LNP",
        ):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        "Could not locate Data/Figure_1d_1e_Spleen_LNP. "
        "Run this notebook inside the supplied NanoSTAMP publication package."
    )


DATA_ROOT = resolve_data_root()
SEG_DIR = DATA_ROOT / "Raw_Spot_Detection_Input"
OUTDIR = DATA_ROOT / "Generated_Output" / "Full_Barcode"
OUTDIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    dict(region="reg000", sample="registered1"),
    dict(region="reg001", sample="registered2"),
]


@dataclass
class Params:
    # Marker-wise LoG candidate generation.
    log_sigma: float = 1.0
    peak_width: int = 21
    nms_min_distance: int = 9
    threshold_peaks: float = 300.0
    initial_calib_threshold: float = 100.0

    # Local signal-to-background extraction.
    snr_core_radius: int = 1
    snr_annulus_inner: int = 3
    snr_annulus_outer: int = 5
    bright_snr_threshold: float = 1.75
    max_bright_markers: int = 9
    min_snr_margin: float = 0.05
    min_on_snr: float = 1.2

    # Decode assumptions.
    expected_on_bits: Optional[int] = None
    barcode_max_hamming_distance: int = 4

    # Hot-pixel masking across barcode channels.
    hotpx_count_threshold: int = 2
    hotpx_sat_value: float = 65500.0

    # Assignment to Mesmer cells from reg*_features.csv centroids.
    cell_assignment_radius_px: float = 25.0
    min_spots_per_cell: int = 1

    # Cell-level binarization.
    min_spots_for_bit: int = 1

    # Optional crop for quick testing.
    roi_yx: Optional[Tuple[slice, slice]] = None


P = Params()
log.info(f'DATA_ROOT:    {DATA_ROOT}')
log.info(f'SEG_DIR:      {SEG_DIR}')
log.info(f'OUTDIR:       {OUTDIR}')
log.info(f'params:       {P}')


## 2. Verify stitched/registered inputs

The detector starts from the two integrated, registered multichannel spleen
stacks. The stitching/registration itself is an upstream image-preparation step;
the registration summaries and marker lists are supplied beside the TIFFs so
reviewers can verify the exact detector inputs.


In [ ]:
def read_marker_list(path: Path) -> List[str]:
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def region_paths(region: str, sample: str) -> Dict[str, Path]:
    return {
        'image': SEG_DIR / f'{sample}_integrated_registered_overlap_crop.tif',
        'markerlist': SEG_DIR / f'{sample}_integrated_MarkerList.txt',
        'features': SEG_DIR / f'{region}_features.csv',
        'tissue_mask': SEG_DIR / f'{region}_mask.npy',
        'summary': SEG_DIR / f'{sample}_registration_summary.json',
    }


region_info = []
for item in REGIONS:
    paths = region_paths(item['region'], item['sample'])
    missing = [name for name, path in paths.items() if name != 'tissue_mask' and not path.exists()]
    if missing:
        raise FileNotFoundError(f"{item['region']} missing: {missing}")

    markers = read_marker_list(paths['markerlist'])
    with open(paths['summary']) as f:
        summary = json.load(f)
    shape = tuple(summary.get('integrated_shape', (len(markers), None, None)))
    barcode_markers = [m for m in markers if m.startswith('A')]
    region_info.append({**item, **paths, 'markers': markers, 'shape': shape, 'barcode_markers': barcode_markers})

pd.DataFrame([
    dict(region=r['region'], sample=r['sample'], image_shape=r['shape'], n_markers=len(r['markers']),
         barcode_markers=', '.join(r['barcode_markers']), features=r['features'].name)
    for r in region_info
])


In [ ]:
# Keep the barcode bit order identical to v3 for direct comparison.
BARCODE_MARKERS = ['A6', 'A17', 'A55', 'A56', 'A76', 'A79', 'A20', 'A46', 'A28', 'A72', 'A2', 'A63']

BARCODE_LIBRARY: Dict[str, str] = {
    '100110100011': 'LNP_A',
}

QC_REFERENCE_LNP = 'LNP_A'


def validate_barcode_library():
    expected_len = len(BARCODE_MARKERS)
    names = list(BARCODE_LIBRARY.values())
    if len(names) != len(set(names)):
        raise ValueError('BARCODE_LIBRARY has duplicated LNP names; names should be unique.')
    for barcode, name in BARCODE_LIBRARY.items():
        if len(barcode) != expected_len:
            raise ValueError(
                f'{name} barcode {barcode} has length {len(barcode)}, '
                f'but BARCODE_MARKERS has length {expected_len}.'
            )
        bad = sorted(set(barcode) - {'0', '1'})
        if bad:
            raise ValueError(f'{name} barcode {barcode} contains non-binary symbols: {bad}')


def marker_sets_for_lnp_name(lnp_name: str) -> Tuple[List[str], List[str]]:
    matches = [barcode for barcode, name in BARCODE_LIBRARY.items() if name == lnp_name]
    if not matches:
        raise ValueError(f'QC_REFERENCE_LNP={lnp_name!r} is not present in BARCODE_LIBRARY')
    barcode = matches[0]
    pos = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '1']
    neg = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '0']
    return pos, neg


validate_barcode_library()
weights = sorted({sum(int(x) for x in barcode) for barcode in BARCODE_LIBRARY})
if len(weights) != 1:
    raise ValueError(f'BARCODE_LIBRARY must be constant-weight for this notebook; got weights {weights}')
EXPECTED_ON_BITS = int(weights[0])
P.expected_on_bits = EXPECTED_ON_BITS
LIBRARY_CODES = list(BARCODE_LIBRARY.keys())
LIBRARY_NAMES = [BARCODE_LIBRARY[k] for k in LIBRARY_CODES]
QC_POSITIVE_MARKERS, QC_NEGATIVE_MARKERS = marker_sets_for_lnp_name(QC_REFERENCE_LNP)

log.info(f'BARCODE_MARKERS ({len(BARCODE_MARKERS)}): {BARCODE_MARKERS}')
log.info(f'Barcode library entries: {len(BARCODE_LIBRARY)} -> {BARCODE_LIBRARY}')
log.info(f'EXPECTED_ON_BITS: {EXPECTED_ON_BITS}')
log.info(f'QC positive markers: {QC_POSITIVE_MARKERS}')
log.info(f'QC negative markers: {QC_NEGATIVE_MARKERS}')


## 3. Helper Functions

The detector now has two separate layers:

1. **Candidate generation**: propose spot coordinates from marker-wise LoG peaks.
2. **Barcode decoding**: at each candidate coordinate, measure all 12 barcode markers,
   call the top `EXPECTED_ON_BITS` as ON, build the full 12-bit barcode, and decode it.

This means a physical spot is treated as one multi-bit object rather than one winning marker.


In [ ]:
def open_stack(path: Path):
    arr = tifffile.memmap(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]
    if arr.ndim != 3:
        raise ValueError(f'Expected image stack as (C,Y,X), got {arr.shape} from {path}')
    return arr


def apply_roi(img2d: np.ndarray, roi_yx):
    if roi_yx is None:
        return img2d
    ys, xs = roi_yx
    return img2d[ys, xs]


def roi_offset(roi_yx):
    if roi_yx is None:
        return 0, 0
    ys, xs = roi_yx
    return (0 if ys.start is None else ys.start, 0 if xs.start is None else xs.start)


def image_qa(img: np.ndarray, label: str) -> dict:
    vals = np.asarray(img)
    finite = vals[np.isfinite(vals)]
    med = float(np.median(finite))
    mad = float(np.median(np.abs(finite - med)))
    p = np.percentile(finite, [1, 99, 99.9, 99.99])
    sat = float((finite >= P.hotpx_sat_value).mean())
    dr = float(p[3] / max(med, 1.0))
    snr = float((p[2] - med) / max(1.4826 * mad, 1.0))
    fg = float((finite > 5 * max(med, 1.0)).mean())
    return dict(label=label, shape=str(vals.shape), dtype=str(vals.dtype), min=float(finite.min()),
                median=med, mean=float(finite.mean()), max=float(finite.max()), p1=float(p[0]),
                p99=float(p[1]), p999=float(p[2]), p9999=float(p[3]), sat_frac=sat,
                dynamic_range=dr, foreground_frac=fg, snr_proxy=snr)


def log_filter(img: np.ndarray, sigma: Optional[float] = None) -> np.ndarray:
    sigma = P.log_sigma if sigma is None else sigma
    return -ndi.gaussian_laplace(img.astype(np.float32, copy=False), sigma=sigma)


def greedy_nms(coords_ij: np.ndarray, scores: np.ndarray, min_distance: float) -> np.ndarray:
    if len(coords_ij) == 0:
        return coords_ij
    order = np.argsort(scores)[::-1]
    coords = coords_ij[order]
    keep = []
    min_d2 = float(min_distance) ** 2
    for pt in coords:
        if not keep:
            keep.append(pt)
            continue
        prev = np.asarray(keep)
        d2 = np.sum((prev - pt) ** 2, axis=1)
        if np.all(d2 >= min_d2):
            keep.append(pt)
    return np.asarray(keep, dtype=np.int64)


def find_spots_from_score(score: np.ndarray, threshold: float) -> pd.DataFrame:
    local_max = score == ndi.maximum_filter(score, size=P.peak_width, mode='nearest')
    above = score >= threshold
    coords = np.argwhere(local_max & above)
    raw_scores = score[coords[:, 0], coords[:, 1]] if len(coords) else np.array([], dtype=np.float32)
    kept = greedy_nms(coords, raw_scores, P.nms_min_distance)
    kept_scores = score[kept[:, 0], kept[:, 1]] if len(kept) else np.array([], dtype=np.float32)
    return pd.DataFrame({'i': kept[:, 0] if len(kept) else [],
                         'j': kept[:, 1] if len(kept) else [],
                         'score': kept_scores})


def load_cell_features(path: Path, roi_yx=None) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'label', 'y', 'x'}
    if not required.issubset(df.columns):
        raise ValueError(f'{path} must contain columns {required}; found {df.columns[:20].tolist()}')
    if roi_yx is not None:
        ys, xs = roi_yx
        y0 = 0 if ys.start is None else ys.start
        y1 = np.inf if ys.stop is None else ys.stop
        x0 = 0 if xs.start is None else xs.start
        x1 = np.inf if xs.stop is None else xs.stop
        df = df[(df['y'] >= y0) & (df['y'] < y1) & (df['x'] >= x0) & (df['x'] < x1)].copy()
        df['y'] -= y0
        df['x'] -= x0
    return df


def assign_spots_to_nearest_cell(spots: pd.DataFrame, cells: pd.DataFrame) -> pd.DataFrame:
    out = spots.copy()
    out['cell'] = 0
    out['cell_distance_px'] = np.nan
    if len(out) == 0 or len(cells) == 0:
        return out
    tree = cKDTree(cells[['y', 'x']].to_numpy(float))
    dist, idx = tree.query(out[['i', 'j']].to_numpy(float), distance_upper_bound=P.cell_assignment_radius_px)
    ok = np.isfinite(dist) & (idx < len(cells))
    labels = cells['label'].to_numpy()
    out.loc[ok, 'cell'] = labels[idx[ok]]
    out.loc[ok, 'cell_distance_px'] = dist[ok]
    return out


def build_hot_mask(stack, region_markers: List[str], marker_to_idx: Dict[str, int], roi_yx):
    first_img = apply_roi(stack[marker_to_idx[region_markers[0]]], roi_yx)
    hot_count = np.zeros(first_img.shape, dtype=np.uint8)
    for marker in region_markers:
        img = apply_roi(stack[marker_to_idx[marker]], roi_yx)
        hot_count += (img >= P.hotpx_sat_value)
    return hot_count >= P.hotpx_count_threshold


def make_ring_masks(core_radius: int, annulus_inner: int, annulus_outer: int):
    r = int(annulus_outer)
    yy, xx = np.meshgrid(np.arange(-r, r + 1), np.arange(-r, r + 1), indexing='ij')
    rr = np.maximum(np.abs(yy), np.abs(xx))
    core_mask = rr <= int(core_radius)
    ann_mask = (rr >= int(annulus_inner)) & (rr <= int(annulus_outer))
    return yy, xx, core_mask, ann_mask


DY, DX, CORE_MASK, ANN_MASK = make_ring_masks(P.snr_core_radius, P.snr_annulus_inner, P.snr_annulus_outer)


def local_signal_background(img: np.ndarray, ii: np.ndarray, jj: np.ndarray):
    h, w = img.shape
    yy = np.clip(ii[:, None, None] + DY[None], 0, h - 1)
    xx = np.clip(jj[:, None, None] + DX[None], 0, w - 1)
    blk = img[yy, xx].astype(np.float32, copy=False)
    signal = blk[:, CORE_MASK].max(axis=1)
    background = np.median(blk[:, ANN_MASK], axis=1)
    return signal, background


def extract_local_snr_tables(stack, markers: List[str], marker_to_idx: Dict[str, int], spots: pd.DataFrame, roi_yx=None):
    vals = pd.DataFrame(index=spots.index)
    if len(spots) == 0:
        return vals, np.zeros((0, len(markers)), dtype=np.float32)
    ii = spots['i'].to_numpy(int)
    jj = spots['j'].to_numpy(int)
    snr_matrix = np.zeros((len(spots), len(markers)), dtype=np.float32)
    for mi, marker in enumerate(markers):
        img = apply_roi(stack[marker_to_idx[marker]], roi_yx)
        signal, background = local_signal_background(img, ii, jj)
        snr = (signal + 1.0) / (background + 1.0)
        vals[f'raw_signal_{marker}'] = signal
        vals[f'raw_background_{marker}'] = background
        vals[f'local_snr_{marker}'] = snr
        vals[f'log_{marker}'] = log_filter(img)[ii, jj]
        snr_matrix[:, mi] = snr
    return vals, snr_matrix


def barcode_to_on_markers(barcode: str, markers: List[str] = None) -> List[str]:
    markers = BARCODE_MARKERS if markers is None else markers
    return [m for m, bit in zip(markers, barcode) if bit == '1']


def marker_on_mask_from_codes(spots: pd.DataFrame, marker: str) -> np.ndarray:
    idx = BARCODE_MARKERS.index(marker)
    return spots['called_code'].str[idx].eq('1').to_numpy() if len(spots) else np.zeros(0, dtype=bool)


def code_from_topk_snr(snr_matrix: np.ndarray, k: int) -> np.ndarray:
    called = np.zeros_like(snr_matrix, dtype=np.int8)
    if snr_matrix.size == 0:
        return called
    topk = np.argsort(snr_matrix, axis=1)[:, -k:]
    rows = np.arange(snr_matrix.shape[0])[:, None]
    called[rows, topk] = 1
    return called


def decode_barcode_to_library(barcode: str) -> dict:
    distances = [(lib_bc, name, sum(a != b for a, b in zip(barcode, lib_bc)))
                 for lib_bc, name in BARCODE_LIBRARY.items()]
    min_dist = min(d for _, _, d in distances)
    candidates = [(lib_bc, name, d) for lib_bc, name, d in distances
                  if d == min_dist and d <= P.barcode_max_hamming_distance]
    if len(candidates) == 1:
        lib_bc, name, dist = candidates[0]
        return dict(matched_barcode=lib_bc, lnp_call=name,
                    barcode_match_distance=dist,
                    barcode_match_status='exact' if dist == 0 else 'tolerant',
                    barcode_in_library=(dist == 0),
                    barcode_excluded=False)
    if len(candidates) > 1:
        return dict(matched_barcode='', lnp_call='ambiguous_mixed',
                    barcode_match_distance=min_dist,
                    barcode_match_status='ambiguous_mixed',
                    barcode_in_library=False,
                    barcode_excluded=True)
    return dict(matched_barcode='', lnp_call='unmapped',
                barcode_match_distance=min_dist,
                barcode_match_status='unmapped',
                barcode_in_library=False,
                barcode_excluded=False)


def decode_spots_full_barcode(stack, region_markers: List[str], marker_to_idx: Dict[str, int], spots: pd.DataFrame, roi_yx=None) -> pd.DataFrame:
    vals, snr_matrix = extract_local_snr_tables(stack, region_markers, marker_to_idx, spots, roi_yx=roi_yx)
    spots = pd.concat([spots.reset_index(drop=True), vals.reset_index(drop=True)], axis=1)
    if len(spots) == 0:
        for col in ['called_code', 'decoded_on_markers', 'min_on_snr', 'max_off_snr', 'snr_margin',
                    'n_bright_markers', 'promiscuous', 'matched_barcode', 'lnp_call',
                    'barcode_match_distance', 'barcode_match_status', 'barcode_in_library',
                    'barcode_excluded', 'pass_quality', 'accepted', 'accept_reason']:
            spots[col] = []
        return spots

    called = code_from_topk_snr(snr_matrix, P.expected_on_bits)
    sorted_snr = np.sort(snr_matrix, axis=1)
    min_on_snr = sorted_snr[:, -P.expected_on_bits]
    max_off_snr = sorted_snr[:, -(P.expected_on_bits + 1)] if snr_matrix.shape[1] > P.expected_on_bits else np.zeros(len(spots))
    snr_margin = min_on_snr - max_off_snr
    n_bright_markers = (snr_matrix > P.bright_snr_threshold).sum(axis=1)
    promiscuous = n_bright_markers > P.max_bright_markers

    codes = [''.join(map(str, row.astype(int))) for row in called]
    decode_rows = [decode_barcode_to_library(code) for code in codes]
    decode_df = pd.DataFrame(decode_rows)

    spots['called_code'] = codes
    spots['decoded_on_markers'] = [', '.join(barcode_to_on_markers(code, region_markers)) for code in codes]
    spots['min_on_snr'] = min_on_snr
    spots['max_off_snr'] = max_off_snr
    spots['snr_margin'] = snr_margin
    spots['n_bright_markers'] = n_bright_markers
    spots['promiscuous'] = promiscuous
    for col in decode_df.columns:
        spots[col] = decode_df[col]
    spots['pass_quality'] = spots['min_on_snr'] >= P.min_on_snr
    spots['accepted'] = (
        spots['pass_quality']
        & (~spots['promiscuous'])
        & (spots['snr_margin'] >= P.min_snr_margin)
        & spots['barcode_match_status'].isin(['exact', 'tolerant'])
    )

    reasons = []
    for _, row in spots.iterrows():
        if row['accepted']:
            reasons.append('accepted')
        elif row['barcode_match_status'] not in ['exact', 'tolerant']:
            reasons.append(f"decode_{row['barcode_match_status']}")
        elif row['promiscuous']:
            reasons.append('promiscuous_multi_marker')
        elif row['min_on_snr'] < P.min_on_snr:
            reasons.append('below_min_on_snr')
        else:
            reasons.append('low_snr_margin')
    spots['accept_reason'] = reasons
    return spots


def merge_marker_candidates(candidate_tables: List[pd.DataFrame]) -> pd.DataFrame:
    if not candidate_tables:
        return pd.DataFrame(columns=['i', 'j', 'score', 'seed_marker'])
    merged = pd.concat(candidate_tables, ignore_index=True)
    if merged.empty:
        return merged
    coords = merged[['i', 'j']].to_numpy(int)
    scores = merged['score'].to_numpy(float)
    kept = greedy_nms(coords, scores, P.nms_min_distance)
    keep_df = pd.DataFrame(kept, columns=['i', 'j'])
    merged = merged.merge(keep_df.assign(_keep=1), on=['i', 'j'], how='inner')
    merged = merged.sort_values('score', ascending=False).drop_duplicates(['i', 'j']).drop(columns='_keep')
    return merged.reset_index(drop=True)


def detect_candidates_in_crop(stack, region_markers: List[str], marker_to_idx: Dict[str, int], y0: int, x0: int,
                              field_size: int, tissue_mask=None) -> pd.DataFrame:
    roi = (slice(y0, y0 + field_size), slice(x0, x0 + field_size))
    hot_mask = build_hot_mask(stack, region_markers, marker_to_idx, roi)
    tissue_crop = tissue_mask[y0:y0 + field_size, x0:x0 + field_size] if tissue_mask is not None else None
    candidate_tables = []
    for marker in region_markers:
        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        score = log_filter(img)
        score[hot_mask] = 0
        if tissue_crop is not None:
            score[~tissue_crop] = 0
        thr = float(PER_MARKER_LOG_THRESHOLDS[marker]['threshold_peaks'])
        tbl = find_spots_from_score(score, thr)
        if len(tbl):
            tbl['seed_marker'] = marker
            candidate_tables.append(tbl)
    merged = merge_marker_candidates(candidate_tables)
    decoded = decode_spots_full_barcode(stack, region_markers, marker_to_idx, merged, roi_yx=roi)
    if len(decoded):
        decoded['i_global'] = decoded['i'] + y0
        decoded['j_global'] = decoded['j'] + x0
    return decoded


def build_cell_bit_count_table(assigned_spots: pd.DataFrame, region_markers: List[str]) -> pd.DataFrame:
    cols = ['cell', *region_markers, 'decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
            'dominant_barcode', 'dominant_barcode_count', 'dominant_lnp_call']
    if len(assigned_spots) == 0:
        return pd.DataFrame(columns=cols)
    assigned = assigned_spots[assigned_spots['cell'] > 0].copy()
    if len(assigned) == 0:
        return pd.DataFrame(columns=cols)

    bit_matrix = np.array([[int(ch) for ch in code] for code in assigned['called_code']], dtype=int)
    bit_df = pd.DataFrame(bit_matrix, columns=region_markers, index=assigned.index)
    bit_df['cell'] = assigned['cell'].to_numpy(int)
    counts = bit_df.groupby('cell')[region_markers].sum().reset_index()

    summary = assigned.groupby('cell').agg(
        decoded_spots=('called_code', 'size'),
        decoded_exact_spots=('barcode_match_status', lambda s: int(np.sum(s == 'exact'))),
        decoded_tolerant_spots=('barcode_match_status', lambda s: int(np.sum(s == 'tolerant'))),
    ).reset_index()

    bc_counts = (assigned.groupby(['cell', 'called_code']).size().rename('n').reset_index()
                 .sort_values(['cell', 'n', 'called_code'], ascending=[True, False, True]))
    dominant_bc = bc_counts.drop_duplicates('cell').rename(columns={'called_code': 'dominant_barcode', 'n': 'dominant_barcode_count'})

    lnp_counts = (assigned.groupby(['cell', 'lnp_call']).size().rename('n').reset_index()
                  .sort_values(['cell', 'n', 'lnp_call'], ascending=[True, False, True]))
    dominant_lnp = lnp_counts.drop_duplicates('cell')[['cell', 'lnp_call']].rename(columns={'lnp_call': 'dominant_lnp_call'})

    out = counts.merge(summary, on='cell', how='left')
    out = out.merge(dominant_bc[['cell', 'dominant_barcode', 'dominant_barcode_count']], on='cell', how='left')
    out = out.merge(dominant_lnp, on='cell', how='left')
    return out


## 4. Calibrate Marker-wise Candidate Thresholds And Barcode-Decode Quality

We still use `reg000` as PBS-negative and `reg001` as positive tissue, but the final quality
threshold is now applied to the **decoded spot** rather than to a single marker.

Calibration steps:

1. choose marker-wise LoG thresholds for candidate generation
2. decode candidate spots in PBS and positive-control fields
3. choose a global `min_on_snr` threshold that keeps positive decoded spots while suppressing PBS decoded spots


In [ ]:
CALIB_NEGATIVE_REGION = 'reg000'
CALIB_POSITIVE_REGION = 'reg001'
CALIB_N_FIELDS_PER_GROUP = 4
CALIB_FIELD_SIZE = 1024
CALIB_RANDOM_SEED = 23
CALIB_MIN_TISSUE_FRACTION = 0.10
CALIB_NEGATIVE_MARKER_PERCENTILE = 99.9
CALIB_MIN_POS_FRACTION = 0.25
CALIB_MAX_NEG_MATCHES_PER_MPX = 3.0
CALIB_MIN_POS_MATCHES_PER_MPX = 0.5


def calib_info_by_region(region: str) -> dict:
    matches = [info for info in region_info if info['region'] == region]
    if not matches:
        raise ValueError(f'Could not find {region!r} in region_info')
    return matches[0]


def sample_tissue_fields(info: dict, n_fields: int, field_size: int, rng) -> List[Tuple[int, int]]:
    h, w = int(info['shape'][1]), int(info['shape'][2])
    field_size = min(int(field_size), h, w)
    tissue_mask = np.load(info['tissue_mask'], mmap_mode='r') if info['tissue_mask'].exists() else None
    fields = []
    max_attempts = max(200, n_fields * 150)
    for _ in range(max_attempts):
        if len(fields) >= n_fields:
            break
        y0 = int(rng.integers(0, h - field_size + 1))
        x0 = int(rng.integers(0, w - field_size + 1))
        if tissue_mask is not None:
            crop_mask = tissue_mask[y0:y0 + field_size, x0:x0 + field_size]
            if float(np.mean(crop_mask)) < CALIB_MIN_TISSUE_FRACTION:
                continue
        fields.append((y0, x0))
    if len(fields) < n_fields:
        log.warning(f"{info['region']}: sampled only {len(fields)}/{n_fields} calibration fields")
    return fields


def marker_peak_table(stack, marker: str, marker_to_idx: Dict[str, int], region_markers: List[str],
                      fields: List[Tuple[int, int]], field_size: int) -> pd.DataFrame:
    rows = []
    for y0, x0 in fields:
        roi = (slice(y0, y0 + field_size), slice(x0, x0 + field_size))
        hot_mask = build_hot_mask(stack, region_markers, marker_to_idx, roi)
        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        score = log_filter(img)
        score[hot_mask] = 0
        peaks = find_spots_from_score(score, P.initial_calib_threshold)
        if len(peaks):
            peaks['field_y0'] = y0
            peaks['field_x0'] = x0
            rows.append(peaks)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['i', 'j', 'score', 'field_y0', 'field_x0'])


def choose_log_threshold(marker: str, neg_tbl: pd.DataFrame, pos_tbl: pd.DataFrame, marker_is_expected_positive: bool) -> dict:
    neg_scores = neg_tbl['score'].to_numpy(float) if len(neg_tbl) else np.array([], dtype=float)
    pos_scores = pos_tbl['score'].to_numpy(float) if len(pos_tbl) else np.array([], dtype=float)

    neg_cons = float(np.percentile(neg_scores, CALIB_NEGATIVE_MARKER_PERCENTILE)) if len(neg_scores) else float(P.threshold_peaks)
    if not marker_is_expected_positive:
        return dict(marker=marker, threshold_peaks=max(neg_cons, float(P.threshold_peaks)), threshold_source='pbs_conservative_off_bit')

    candidates = np.unique(np.round(np.r_[neg_scores, pos_scores, [neg_cons, P.threshold_peaks]], 3))
    candidates = candidates[np.isfinite(candidates) & (candidates > 0)]
    if candidates.size == 0:
        candidates = np.asarray([float(P.threshold_peaks)])

    best_row = None
    best_score = None
    for thr in candidates:
        neg_pass = float(np.mean(neg_scores >= thr)) if len(neg_scores) else 0.0
        pos_pass = float(np.mean(pos_scores >= thr)) if len(pos_scores) else 0.0
        score = (pos_pass - neg_pass) + 2.0 * (pos_pass >= CALIB_MIN_POS_FRACTION)
        row = dict(marker=marker, threshold_peaks=float(thr), threshold_source='pbs_vs_positive_on_bit',
                   neg_fraction_ge_threshold=neg_pass, pos_fraction_ge_threshold=pos_pass)
        if best_score is None or score > best_score:
            best_score = score
            best_row = row
    return best_row


def decode_candidate_fields(stack, region_markers: List[str], marker_to_idx: Dict[str, int], fields: List[Tuple[int, int]],
                            field_size: int, tissue_mask=None) -> pd.DataFrame:
    rows = []
    for y0, x0 in fields:
        decoded = detect_candidates_in_crop(stack, region_markers, marker_to_idx, y0, x0, field_size, tissue_mask=tissue_mask)
        if len(decoded):
            decoded['field_y0'] = y0
            decoded['field_x0'] = x0
            rows.append(decoded)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def calibrate_decode_thresholds(neg_decoded: pd.DataFrame, pos_decoded: pd.DataFrame, neg_area_mpx: float, pos_area_mpx: float):
    if len(neg_decoded) == 0 and len(pos_decoded) == 0:
        return pd.DataFrame([dict(min_on_snr=P.min_on_snr, neg_matches_per_mpx=0.0, pos_matches_per_mpx=0.0,
                                  enrichment=np.nan, passes_policy=False, selection_score=-np.inf)])

    vals = []
    for df in [neg_decoded, pos_decoded]:
        if len(df):
            vals.extend(df['min_on_snr'].tolist())
    vals.extend([1.0, 1.2, 1.5, 1.8, 2.0, 2.5, 3.0])
    thr_grid = np.unique(np.round([v for v in vals if np.isfinite(v) and v > 0], 3))

    rows = []
    for thr in thr_grid:
        neg_keep = neg_decoded[
            neg_decoded['barcode_match_status'].isin(['exact', 'tolerant'])
            & (~neg_decoded['promiscuous'])
            & (neg_decoded['snr_margin'] >= P.min_snr_margin)
            & (neg_decoded['min_on_snr'] >= thr)
        ] if len(neg_decoded) else neg_decoded
        pos_keep = pos_decoded[
            pos_decoded['barcode_match_status'].isin(['exact', 'tolerant'])
            & (~pos_decoded['promiscuous'])
            & (pos_decoded['snr_margin'] >= P.min_snr_margin)
            & (pos_decoded['min_on_snr'] >= thr)
        ] if len(pos_decoded) else pos_decoded

        neg_rate = len(neg_keep) / max(neg_area_mpx, 1e-9)
        pos_rate = len(pos_keep) / max(pos_area_mpx, 1e-9)
        enrichment = (pos_rate + 0.1) / (neg_rate + 0.1)
        passes_policy = (neg_rate <= CALIB_MAX_NEG_MATCHES_PER_MPX) and (pos_rate >= CALIB_MIN_POS_MATCHES_PER_MPX)
        selection_score = (np.log1p(pos_rate) * np.log1p(enrichment)) - np.log1p(neg_rate)
        if passes_policy:
            selection_score += 1000.0
        rows.append(dict(min_on_snr=float(thr), neg_matches_per_mpx=neg_rate, pos_matches_per_mpx=pos_rate,
                         enrichment=enrichment, passes_policy=passes_policy, selection_score=selection_score,
                         n_neg_keep=int(len(neg_keep)), n_pos_keep=int(len(pos_keep))))
    return pd.DataFrame(rows).sort_values(['selection_score', 'pos_matches_per_mpx', 'enrichment'], ascending=False).reset_index(drop=True)


def run_v4_calibration():
    neg_info = calib_info_by_region(CALIB_NEGATIVE_REGION)
    pos_info = calib_info_by_region(CALIB_POSITIVE_REGION)
    rng = np.random.default_rng(CALIB_RANDOM_SEED)
    field_size = min(CALIB_FIELD_SIZE, int(neg_info['shape'][1]), int(neg_info['shape'][2]), int(pos_info['shape'][1]), int(pos_info['shape'][2]))

    neg_stack = open_stack(neg_info['image'])
    pos_stack = open_stack(pos_info['image'])
    neg_marker_to_idx = {m: i for i, m in enumerate(neg_info['markers'])}
    pos_marker_to_idx = {m: i for i, m in enumerate(pos_info['markers'])}
    markers = [m for m in BARCODE_MARKERS if m in neg_marker_to_idx and m in pos_marker_to_idx]
    expected_positive_markers = set(marker_sets_for_lnp_name(QC_REFERENCE_LNP)[0])

    neg_fields = sample_tissue_fields(neg_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)
    pos_fields = sample_tissue_fields(pos_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)

    log_rows = []
    for marker in markers:
        neg_tbl = marker_peak_table(neg_stack, marker, neg_marker_to_idx, markers, neg_fields, field_size)
        pos_tbl = marker_peak_table(pos_stack, marker, pos_marker_to_idx, markers, pos_fields, field_size)
        log_rows.append(choose_log_threshold(marker, neg_tbl, pos_tbl, marker in expected_positive_markers))
    log_df = pd.DataFrame(log_rows).sort_values('marker').reset_index(drop=True)

    global PER_MARKER_LOG_THRESHOLDS
    PER_MARKER_LOG_THRESHOLDS = {
        row['marker']: dict(threshold_peaks=float(row['threshold_peaks']), threshold_source=row['threshold_source'])
        for _, row in log_df.iterrows()
    }

    neg_tissue_mask = np.load(neg_info['tissue_mask'], mmap_mode='r').astype(bool) if neg_info['tissue_mask'].exists() else None
    pos_tissue_mask = np.load(pos_info['tissue_mask'], mmap_mode='r').astype(bool) if pos_info['tissue_mask'].exists() else None
    neg_decoded = decode_candidate_fields(neg_stack, markers, neg_marker_to_idx, neg_fields, field_size, tissue_mask=neg_tissue_mask)
    pos_decoded = decode_candidate_fields(pos_stack, markers, pos_marker_to_idx, pos_fields, field_size, tissue_mask=pos_tissue_mask)

    neg_area_mpx = len(neg_fields) * field_size * field_size / 1e6
    pos_area_mpx = len(pos_fields) * field_size * field_size / 1e6
    decode_df = calibrate_decode_thresholds(neg_decoded, pos_decoded, neg_area_mpx, pos_area_mpx)
    best = decode_df.iloc[0]
    P.min_on_snr = float(best['min_on_snr'])

    global SELECTED_DETECTION_PARAMS
    SELECTED_DETECTION_PARAMS = {
        'source': 'v4_full_barcode_decode',
        'log_sigma': P.log_sigma,
        'peak_width': P.peak_width,
        'nms_min_distance': P.nms_min_distance,
        'expected_on_bits': P.expected_on_bits,
        'min_on_snr': P.min_on_snr,
        'min_snr_margin': P.min_snr_margin,
        'bright_snr_threshold': P.bright_snr_threshold,
        'max_bright_markers': P.max_bright_markers,
        'per_marker_log_thresholds': PER_MARKER_LOG_THRESHOLDS,
    }

    log_df.to_csv(OUTDIR / 'calibrated_marker_log_thresholds_v4.csv', index=False)
    decode_df.to_csv(OUTDIR / 'calibrated_decode_thresholds_v4.csv', index=False)
    (OUTDIR / 'selected_detection_parameters_v4.json').write_text(json.dumps(SELECTED_DETECTION_PARAMS, indent=2))
    log.info(f"Selected min_on_snr={P.min_on_snr:.3f}; PBS={best['neg_matches_per_mpx']:.2f} matches/Mpx, positive={best['pos_matches_per_mpx']:.2f} matches/Mpx")
    return log_df, decode_df, neg_decoded, pos_decoded


marker_log_thresholds_v4, decode_thresholds_v4, calib_neg_decoded_v4, calib_pos_decoded_v4 = run_v4_calibration()
display(marker_log_thresholds_v4)
display(decode_thresholds_v4.head(20))


## 5. Run full-barcode spot detection for the two spleen regions

For each region:

1. propose marker-wise LoG peak candidates;
2. merge nearby candidate coordinates;
3. measure local signal-to-background across all 12 barcode channels;
4. call the six strongest bits ON;
5. decode the complete 12-bit word against the SM-102 LNP codebook;
6. retain spots passing barcode and local-SNR quality criteria; and
7. assign accepted spots to the nearest supplied cell centroid.

The following cell is computationally intensive because it reads the full
registered TIFF stacks. It was not run while preparing this publication copy.


In [ ]:
all_spots_v4 = {}
all_accepted_spots_v4 = {}
all_cells_v4 = {}
qa_rows_v4 = []
spot_qa_rows_v4 = []

for r in region_info:
    region = r['region']
    sample = r['sample']
    log.info(f'===== {region} / {sample} =====')
    t_region = tic()

    markers = r['markers']
    marker_to_idx = {m: i for i, m in enumerate(markers)}
    region_markers = [m for m in BARCODE_MARKERS if m in marker_to_idx]
    stack = open_stack(r['image'])
    log.info(f'image shape: {stack.shape}; barcode markers: {region_markers}')

    hot_mask = build_hot_mask(stack, region_markers, marker_to_idx, P.roi_yx)
    n_hot = int(hot_mask.sum())
    log.info(f'hot pixels masked: {n_hot}')

    tissue_mask = None
    if r['tissue_mask'].exists():
        tissue_mask = apply_roi(np.load(r['tissue_mask'], mmap_mode='r').astype(bool), P.roi_yx)

    for marker in region_markers:
        img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
        qa_rows_v4.append({'region': region, 'sample': sample, 'marker': marker, **image_qa(img, marker)})

    candidate_tables = []
    for marker in region_markers:
        img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
        score = log_filter(img)
        score[hot_mask] = 0
        if tissue_mask is not None and tissue_mask.shape == score.shape:
            score[~tissue_mask] = 0
        thr = float(PER_MARKER_LOG_THRESHOLDS[marker]['threshold_peaks'])
        spots = find_spots_from_score(score, thr)
        if len(spots):
            spots['seed_marker'] = marker
            candidate_tables.append(spots)

    merged = merge_marker_candidates(candidate_tables)
    spots = decode_spots_full_barcode(stack, region_markers, marker_to_idx, merged, roi_yx=P.roi_yx)
    y0, x0 = roi_offset(P.roi_yx)
    if len(spots):
        spots['i_global'] = spots['i'] + y0
        spots['j_global'] = spots['j'] + x0

    accepted = spots[spots['accepted']].copy() if len(spots) else spots.copy()
    cells = load_cell_features(r['features'], roi_yx=P.roi_yx)
    accepted = assign_spots_to_nearest_cell(accepted, cells)
    counts = build_cell_bit_count_table(accepted, region_markers)
    if len(counts):
        counts['total_spots'] = counts['decoded_spots']
        counts = counts[counts['total_spots'] >= P.min_spots_per_cell].reset_index(drop=True)
    else:
        counts = pd.DataFrame(columns=['cell', *region_markers, 'decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
                                       'dominant_barcode', 'dominant_barcode_count', 'dominant_lnp_call', 'total_spots'])

    spots.to_csv(OUTDIR / f'spots_all_candidates_{region}_v4.csv', index=False)
    accepted.to_csv(OUTDIR / f'spots_{region}_v4.csv', index=False)
    counts.to_csv(OUTDIR / f'cell_spot_counts_{region}_v4.csv', index=False)

    all_spots_v4[region] = spots
    all_accepted_spots_v4[region] = accepted
    all_cells_v4[region] = counts
    spot_qa_rows_v4.append(dict(
        region=region,
        sample=sample,
        n_markers=len(region_markers),
        detection_source='v4_full_barcode_decode',
        n_hot_pixels=n_hot,
        n_candidates=len(spots),
        n_spots=len(accepted),
        n_spots_assigned=int((accepted['cell'] > 0).sum()) if len(accepted) else 0,
        n_callable_cells=len(counts),
        n_exact=int(np.sum(accepted['barcode_match_status'] == 'exact')) if len(accepted) else 0,
        n_tolerant=int(np.sum(accepted['barcode_match_status'] == 'tolerant')) if len(accepted) else 0,
        median_min_on_snr=float(np.median(accepted['min_on_snr'])) if len(accepted) else np.nan,
        median_snr_margin=float(np.median(accepted['snr_margin'])) if len(accepted) else np.nan,
        frac_promiscuous=float(spots['promiscuous'].mean()) if len(spots) else np.nan,
    ))
    toc(t_region, f'{region} total')

qa_per_image_v4 = pd.DataFrame(qa_rows_v4)
qa_spot_detection_v4 = pd.DataFrame(spot_qa_rows_v4)
qa_per_image_v4.to_csv(OUTDIR / 'qa_per_image_v4.csv', index=False)
qa_spot_detection_v4.to_csv(OUTDIR / 'qa_spot_detection_v4.csv', index=False)
qa_spot_detection_v4


## 6. Combine spleen outputs and write CSV summaries

Only tabular outputs are written. The exploratory PNG-producing QC and showcase
sections from the working notebook were removed from this publication copy.


In [ ]:
combined_spots_v4 = []
combined_cells_v4 = []
for region, df in all_accepted_spots_v4.items():
    tmp = df.copy()
    tmp.insert(0, 'region', region)
    combined_spots_v4.append(tmp)
for region, df in all_cells_v4.items():
    tmp = df.copy()
    tmp.insert(0, 'region', region)
    combined_cells_v4.append(tmp)

combined_spots_v4 = pd.concat(combined_spots_v4, ignore_index=True) if combined_spots_v4 else pd.DataFrame()
combined_cells_v4 = pd.concat(combined_cells_v4, ignore_index=True) if combined_cells_v4 else pd.DataFrame()
combined_spots_v4.to_csv(OUTDIR / 'spots_all_regions_v4.csv', index=False)
combined_cells_v4.to_csv(OUTDIR / 'cell_spot_counts_all_regions_v4.csv', index=False)

summary_v4 = (combined_spots_v4.groupby(['region', 'lnp_call']).size().rename('n_spots').reset_index())
summary_v4.to_csv(OUTDIR / 'decoded_spot_counts_by_lnp_v4.csv', index=False)
summary_v4


## 7. Build the final cell-level codebook table

Accepted decoded spots are aggregated per cell. Each accepted physical RCA spot
contributes all six decoded ON bits, after which cells are called codebook-matched
LNP-positive. This table is the codebook input used by the Figure 1d/1e analysis.


In [ ]:
def hamming_distance(a: str, b: str) -> int:
    if len(a) != len(b):
        return max(len(a), len(b))
    return sum(x != y for x, y in zip(a, b))


def tolerant_lnp_match(barcode: str, total_spots: int) -> dict:
    if total_spots == 0:
        return dict(lnp_call='no_barcode', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='no_barcode',
                    barcode_excluded=True)
    if not BARCODE_LIBRARY:
        return dict(lnp_call='unmapped', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='unmapped',
                    barcode_excluded=False)

    distances = [(lib_bc, name, hamming_distance(barcode, lib_bc))
                 for lib_bc, name in BARCODE_LIBRARY.items()]
    min_dist = min(d for _, _, d in distances)
    candidates = [(lib_bc, name, d) for lib_bc, name, d in distances
                  if d == min_dist and d <= P.barcode_max_hamming_distance]

    if len(candidates) == 1:
        lib_bc, name, dist = candidates[0]
        return dict(lnp_call=name, barcode_in_library=(dist == 0),
                    barcode_match_distance=dist,
                    barcode_match_status='exact' if dist == 0 else 'tolerant',
                    barcode_excluded=False)
    if len(candidates) > 1:
        return dict(lnp_call='ambiguous_mixed', barcode_in_library=False,
                    barcode_match_distance=min_dist, barcode_match_status='ambiguous_mixed',
                    barcode_excluded=True)
    return dict(lnp_call='unmapped', barcode_in_library=False,
                barcode_match_distance=min_dist, barcode_match_status='unmapped',
                barcode_excluded=False)


def build_cell_analysis_table_for_region(region: str, info: dict, spots: pd.DataFrame) -> pd.DataFrame:
    cells = pd.read_csv(info['features'])
    if 'label' not in cells.columns or 'x' not in cells.columns or 'y' not in cells.columns:
        raise ValueError(f"{info['features']} must contain label, x, and y columns")

    cells = cells.copy()
    cells.insert(0, 'region', region)
    cells = cells.rename(columns={'label': 'cell'})

    assigned = spots[spots['cell'] > 0].copy() if len(spots) else pd.DataFrame()
    if len(assigned):
        bit_matrix = np.array([[int(ch) for ch in code] for code in assigned['called_code']], dtype=int)
        counts = pd.DataFrame(bit_matrix, columns=BARCODE_MARKERS, index=assigned.index)
        counts['cell'] = assigned['cell'].to_numpy(int)
        counts = counts.groupby('cell')[BARCODE_MARKERS].sum().reset_index()
        counts.columns = ['cell', *[f'spot_{m}' for m in BARCODE_MARKERS]]

        decoded_summary = assigned.groupby('cell').agg(
            decoded_spots=('called_code', 'size'),
            decoded_exact_spots=('barcode_match_status', lambda s: int(np.sum(s == 'exact'))),
            decoded_tolerant_spots=('barcode_match_status', lambda s: int(np.sum(s == 'tolerant'))),
        ).reset_index()

        per_bc = (assigned.groupby(['cell', 'called_code']).size().rename('n').reset_index()
                  .sort_values(['cell', 'n', 'called_code'], ascending=[True, False, True]))
        dominant_bc = per_bc.drop_duplicates('cell').rename(columns={'called_code': 'dominant_decoded_barcode', 'n': 'dominant_decoded_barcode_count'})
    else:
        counts = pd.DataFrame({'cell': cells['cell']})
        for marker in BARCODE_MARKERS:
            counts[f'spot_{marker}'] = 0
        decoded_summary = pd.DataFrame({'cell': cells['cell'], 'decoded_spots': 0, 'decoded_exact_spots': 0, 'decoded_tolerant_spots': 0})
        dominant_bc = pd.DataFrame({'cell': cells['cell'], 'dominant_decoded_barcode': '', 'dominant_decoded_barcode_count': 0})

    out = cells.merge(counts, on='cell', how='left')
    out = out.merge(decoded_summary, on='cell', how='left')
    out = out.merge(dominant_bc[['cell', 'dominant_decoded_barcode', 'dominant_decoded_barcode_count']], on='cell', how='left')

    spot_cols = [f'spot_{m}' for m in BARCODE_MARKERS]
    fill_zero_cols = spot_cols + ['decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots', 'dominant_decoded_barcode_count']
    for col in fill_zero_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = out[col].fillna(0)
    for col in spot_cols + ['decoded_spots', 'decoded_exact_spots', 'decoded_tolerant_spots', 'dominant_decoded_barcode_count']:
        out[col] = out[col].astype(int)
    out['dominant_decoded_barcode'] = out['dominant_decoded_barcode'].fillna('')

    bit_cols = []
    for marker in BARCODE_MARKERS:
        bit_col = f'bit_{marker}'
        spot_col = f'spot_{marker}'
        out[bit_col] = (out[spot_col] >= P.min_spots_for_bit).astype(int)
        bit_cols.append(bit_col)

    out['barcode'] = out[bit_cols].astype(str).agg(''.join, axis=1)
    out['total_barcode_spots'] = out['decoded_spots']
    out['n_positive_bits'] = out[bit_cols].sum(axis=1)

    spot_matrix = out[spot_cols].to_numpy(int)
    if len(out):
        dominant_idx = np.argmax(spot_matrix, axis=1)
        dominant_counts = spot_matrix[np.arange(len(out)), dominant_idx]
        out['dominant_barcode_marker'] = [BARCODE_MARKERS[i] if total > 0 else 'none'
                                          for i, total in zip(dominant_idx, out['total_barcode_spots'])]
        out['dominant_barcode_count'] = dominant_counts
        out['dominant_barcode_fraction'] = np.where(
            out['total_barcode_spots'] > 0,
            out['dominant_barcode_count'] / out['total_barcode_spots'],
            0.0,
        )
    else:
        out['dominant_barcode_marker'] = []
        out['dominant_barcode_count'] = []
        out['dominant_barcode_fraction'] = []

    positive_spot_sum = np.zeros(len(out), dtype=int)
    for marker in BARCODE_MARKERS:
        positive_spot_sum += out[f'spot_{marker}'].where(out[f'bit_{marker}'] == 1, 0).to_numpy(int)
    out['barcode_confidence'] = np.where(
        out['total_barcode_spots'] > 0,
        positive_spot_sum / (out['total_barcode_spots'] * max(P.expected_on_bits, 1)),
        0.0,
    )

    match_rows = [tolerant_lnp_match(bc, int(total))
                  for bc, total in zip(out['barcode'], out['total_barcode_spots'])]
    match_df = pd.DataFrame(match_rows, index=out.index)
    for col in match_df.columns:
        out[col] = match_df[col]
    out['lnp_positive'] = (
        (~out['barcode_excluded'])
        & ~out['lnp_call'].isin(['no_barcode', 'unmapped', 'ambiguous_mixed'])
    )

    front = [
        'region', 'cell', 'x', 'y', 'area', 'eccentricity',
        'barcode', 'lnp_call', 'lnp_positive', 'barcode_in_library',
        'barcode_match_distance', 'barcode_match_status', 'barcode_excluded',
        'total_barcode_spots', 'decoded_exact_spots', 'decoded_tolerant_spots',
        'n_positive_bits', 'dominant_decoded_barcode', 'dominant_decoded_barcode_count',
        'dominant_barcode_marker', 'dominant_barcode_count', 'dominant_barcode_fraction',
        'barcode_confidence',
    ]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]


cell_analysis_tables_v4 = []
for info in region_info:
    region = info['region']
    table = build_cell_analysis_table_for_region(region, info, all_accepted_spots_v4.get(region, pd.DataFrame()))
    table.to_csv(OUTDIR / f'cell_analysis_table_{region}_v4.csv', index=False)
    cell_analysis_tables_v4.append(table)
    log.info(f"{region}: wrote cell_analysis_table_{region}_v4.csv with {len(table)} cells")

cell_analysis_table_v4 = pd.concat(cell_analysis_tables_v4, ignore_index=True)
cell_analysis_table_v4.to_csv(OUTDIR / 'cell_analysis_table_spleen_regions_v4.csv', index=False)
log.info(f"Wrote {OUTDIR / 'cell_analysis_table_spleen_regions_v4.csv'} with {len(cell_analysis_table_v4)} cells")

cell_analysis_summary_v4 = (cell_analysis_table_v4
                            .groupby(['region', 'lnp_call'], dropna=False)
                            .size()
                            .reset_index(name='n_cells'))
cell_analysis_summary_v4.to_csv(OUTDIR / 'cell_analysis_summary_by_lnp_call_v4.csv', index=False)

print('Full 12-bit barcode summary (v4):')
display(cell_analysis_summary_v4)


## CSV outputs

The principal files written to `Generated_Output/` are:

- `spots_reg000_v4.csv` and `spots_reg001_v4.csv`: accepted decoded spots;
- `cell_spot_counts_reg000_v4.csv` and `cell_spot_counts_reg001_v4.csv`:
  decoded-bit counts per assigned cell;
- `spots_all_regions_v4.csv`: accepted spots combined across the two spleens;
- `cell_spot_counts_all_regions_v4.csv`: combined cell spot-count table; and
- `cell_analysis_table_spleen_regions_v4.csv`: final cell-level codebook calls.

The precomputed, compact Figure 1d/1e inputs are supplied separately so the
downstream plotting notebook can run without rerunning this large-image step.
